<a href="https://www.kaggle.com/code/asivakumarnair/diabetic-retinopathy-imagenet?scriptVersionId=344745218" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
# ===== STAGE 16 SESSION: FOLD 1, MOBILENETV2 + RESNET50 (loads locked fold file, never regenerates) =====

!pip install -q tensorflow==2.19.0

import os
os.environ['TF_USE_LEGACY_KERAS'] = '1'

import random
import numpy as np
import pandas as pd
import tensorflow as tf

SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
os.environ['TF_DETERMINISTIC_OPS'] = '1'
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
print(f"Seed {SEED} set, TF {tf.__version__}, tf.keras module: {tf.keras.__name__}")
assert 'tf_keras' in tf.keras.__name__, "STOP: Keras 3 active, not legacy. Restart before continuing."

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout
from tensorflow.keras.applications import MobileNetV2, ResNet50
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input as mob_pre
from tensorflow.keras.applications.resnet50 import preprocess_input as res_pre
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, CSVLogger
from sklearn.utils.class_weight import compute_class_weight
from sklearn.model_selection import train_test_split
from sklearn.metrics import (cohen_kappa_score, roc_auc_score, accuracy_score,
                              f1_score, recall_score, confusion_matrix)

# ---------- LOAD LOCKED FOLD ASSIGNMENTS, DO NOT REGENERATE ----------
FOLD_CSV_PATH = '/kaggle/input/datasets/asivakumarnair/drcvfoldassignments/dr_cv_fold_assignments.csv'   # the dataset you uploaded after the custom/eff session

GRADES, NUM_CLASSES = ['0','1','2','3','4'], 5
IMG_SIZE, BATCH_SIZE = 224, 32
PHASE1_EPOCHS, PHASE1_LR, PHASE2_LR, EARLYSTOP_PAT = 10, 1e-3, 1e-5, 7
AUG = dict(rotation_range=20, width_shift_range=0.1, height_shift_range=0.1,
           horizontal_flip=True, zoom_range=0.1)
CURRENT_FOLD = 1

pooled = pd.read_csv(FOLD_CSV_PATH)
pooled['grade'] = pooled['grade'].astype(str)
print(f"Loaded {len(pooled):,} rows (expect 9,068)")
assert len(pooled) == 9068, "Row count mismatch, wrong file or corrupted upload"
assert pooled['fold'].nunique() == 5, "Fold file does not have 5 folds"
print(pooled['fold'].value_counts().sort_index())

test_df   = pooled[pooled.fold == CURRENT_FOLD].reset_index(drop=True)
remainder = pooled[pooled.fold != CURRENT_FOLD].reset_index(drop=True)

def safe_split(df, label_col, test_size, rs, tag=""):
    try:
        return train_test_split(df, test_size=test_size, stratify=df[label_col], random_state=rs)
    except ValueError as e:
        print(f"WARNING [{tag}]: stratified split failed, unstratified fallback. {e}")
        return train_test_split(df, test_size=test_size, random_state=rs)

def split_group_level_inner(df, label_col='grade', val_frac=0.15, rs=SEED, tag=""):
    grouped = df.groupby('group_id')[label_col].agg(lambda s: s.value_counts().index[0]).reset_index()
    g_tr, g_va = safe_split(grouped, label_col, val_frac, rs, tag=tag)
    pick = lambda ids: df[df['group_id'].isin(ids['group_id'])]
    return pick(g_tr), pick(g_va)

train_parts, val_parts = [], []
for src in ['aptos', 'eyepacs', 'messidor']:
    sub = remainder[remainder.source == src]
    tr_s, va_s = split_group_level_inner(sub, tag=f"fold{CURRENT_FOLD}-{src}-inner")
    train_parts.append(tr_s); val_parts.append(va_s)
train_df = pd.concat(train_parts, ignore_index=True)
val_df   = pd.concat(val_parts, ignore_index=True)

print(f"\nFold {CURRENT_FOLD}: Train {len(train_df):,} | Val {len(val_df):,} | Test {len(test_df):,}")
print("Must reproduce the custom/eff session exactly: Train 6,165 | Val 1,092 | Test 1,811")
assert len(train_df) == 6165 and len(val_df) == 1092 and len(test_df) == 1811, \
    "Split sizes do not match the custom/eff session, the loaded file may not be the same one, stop and investigate"

for src in ['aptos', 'eyepacs', 'messidor']:
    tr_g = set(train_df[train_df.source==src]['group_id'])
    va_g = set(val_df[val_df.source==src]['group_id'])
    te_g = set(test_df[test_df.source==src]['group_id'])
    ok = tr_g.isdisjoint(va_g) and tr_g.isdisjoint(te_g) and va_g.isdisjoint(te_g)
    print(f"  {src}: train/val/test group-disjoint = {ok}")
    assert ok, f"LEAKAGE in fold {CURRENT_FOLD}, source {src}"
print(f"Fold {CURRENT_FOLD} leakage check: PASS")

cls = np.array(GRADES)
cw = compute_class_weight('balanced', classes=cls, y=train_df['grade'])
CLASS_WEIGHT = {i: w for i, w in enumerate(cw)}
print(f"Fold {CURRENT_FOLD} class_weight:", {c: round(w,3) for c,w in zip(cls,cw)}, f"| span {cw.max()/cw.min():.1f}x")
print("Must reproduce the custom/eff session: span 15.9x")

def make_gens(preprocess_fn):
    train_idg = ImageDataGenerator(preprocessing_function=preprocess_fn, **AUG)
    eval_idg  = ImageDataGenerator(preprocessing_function=preprocess_fn)
    common = dict(x_col='image_path', y_col='grade', target_size=(IMG_SIZE,IMG_SIZE), batch_size=BATCH_SIZE,
                  class_mode='categorical', classes=GRADES, color_mode='rgb')
    return (train_idg.flow_from_dataframe(train_df, shuffle=True, seed=SEED, **common),
            eval_idg.flow_from_dataframe(val_df, shuffle=False, **common),
            eval_idg.flow_from_dataframe(test_df, shuffle=False, **common))

def build_pretrained(base_class, num_classes=5, shape=(224,224,3)):
    base = base_class(include_top=False, weights='imagenet', input_shape=shape)
    model = Sequential([base, GlobalAveragePooling2D(), Dense(256,activation='relu'),
                        Dropout(0.3), Dense(num_classes,activation='softmax')])
    return model, base

def macro_specificity(y_true, y_pred, n_classes=NUM_CLASSES):
    cm = confusion_matrix(y_true, y_pred, labels=range(n_classes))
    total = cm.sum(); specs = []
    for i in range(n_classes):
        tp = cm[i,i]; fn = cm[i,:].sum()-tp; fp = cm[:,i].sum()-tp
        tn = total-tp-fn-fp
        specs.append(tn/(tn+fp) if (tn+fp) > 0 else np.nan)
    return np.nanmean(specs)

def full_test_metrics(model, te_gen):
    y_prob = model.predict(te_gen, verbose=0)
    y_true = np.asarray(te_gen.classes)
    y_pred = y_prob.argmax(axis=1)
    try:
        auc = roc_auc_score(np.eye(NUM_CLASSES)[y_true], y_prob, average='macro', multi_class='ovr')
    except ValueError:
        auc = np.nan
    return dict(qwk=cohen_kappa_score(y_true, y_pred, weights='quadratic'), macro_auc=auc,
                accuracy=accuracy_score(y_true, y_pred),
                macro_f1=f1_score(y_true, y_pred, average='macro'),
                macro_sensitivity=recall_score(y_true, y_pred, average='macro'),
                macro_specificity=macro_specificity(y_true, y_pred), n_test=len(y_true)), y_true, y_pred, y_prob

def train_and_evaluate(arch_code, base_class, preprocess_fn):
    tr, va, te = make_gens(preprocess_fn)
    auc_path = f'/kaggle/working/cv_f{CURRENT_FOLD}_{arch_code}_aucbest.keras'
    acc_path = f'/kaggle/working/cv_f{CURRENT_FOLD}_{arch_code}_accbest.keras'

    model, base = build_pretrained(base_class)
    base.trainable = False
    model.compile(Adam(PHASE1_LR), 'categorical_crossentropy',
                  metrics=['accuracy', tf.keras.metrics.AUC(name='auc', multi_label=False)])
    print(f"\n===== Fold {CURRENT_FOLD}, {arch_code}: PHASE 1 (head only, {PHASE1_EPOCHS} epochs) =====")
    model.fit(tr, validation_data=va, epochs=PHASE1_EPOCHS, class_weight=CLASS_WEIGHT, verbose=1,
              callbacks=[CSVLogger(f'/kaggle/working/cv_f{CURRENT_FOLD}_{arch_code}_log.csv', append=False)])
    base.trainable = True
    model.compile(Adam(PHASE2_LR), 'categorical_crossentropy',
                  metrics=['accuracy', tf.keras.metrics.AUC(name='auc', multi_label=False)])
    print(f"\n===== Fold {CURRENT_FOLD}, {arch_code}: PHASE 2 (full fine-tune, dual checkpoint) =====")
    hist = model.fit(tr, validation_data=va, epochs=60, class_weight=CLASS_WEIGHT, verbose=1,
              callbacks=[EarlyStopping(monitor='val_auc', mode='max', patience=EARLYSTOP_PAT, restore_best_weights=True),
                         ModelCheckpoint(auc_path, monitor='val_auc', mode='max', save_best_only=True),
                         ModelCheckpoint(acc_path, monitor='val_accuracy', mode='max', save_best_only=True),
                         CSVLogger(f'/kaggle/working/cv_f{CURRENT_FOLD}_{arch_code}_log.csv', append=True)])

    live_metrics, y_true, y_pred, y_prob = full_test_metrics(model, te)
    print(f"\nLive (in-memory) test metrics, auc-selected: {live_metrics}")
    np.savez(f'/kaggle/working/cv_f{CURRENT_FOLD}_{arch_code}_preds.npz',
             y_true=y_true, y_pred=y_pred, y_prob=y_prob,
             source=test_df['source'].values, group_id=test_df['group_id'].values)

    reloaded_auc_model = load_model(auc_path)
    reloaded_auc_metrics, _, _, _ = full_test_metrics(reloaded_auc_model, te)
    match = abs(live_metrics['macro_auc'] - reloaded_auc_metrics['macro_auc']) < 1e-3
    print(f"Reloaded auc-checkpoint macro_auc={reloaded_auc_metrics['macro_auc']:.4f} vs live={live_metrics['macro_auc']:.4f}, match={match}")
    assert match, "AUC-CHECKPOINT MISMATCH, invalid provenance, discard this result"
    del reloaded_auc_model

    best_val_acc_seen = max(hist.history['val_accuracy'])
    reloaded_acc_model = load_model(acc_path)
    reloaded_val_acc = reloaded_acc_model.evaluate(va, verbose=0)[1]
    match_acc = abs(best_val_acc_seen - reloaded_val_acc) < 1e-3
    print(f"Accuracy-checkpoint provenance: best val_accuracy seen during training={best_val_acc_seen:.4f} "
          f"vs reloaded val_accuracy={reloaded_val_acc:.4f}, match={match_acc}")
    assert match_acc, "ACCURACY-CHECKPOINT MISMATCH, invalid provenance, discard this result"
    reloaded_acc_metrics, _, _, _ = full_test_metrics(reloaded_acc_model, te)
    del reloaded_acc_model
    del model, base; import gc; gc.collect(); tf.keras.backend.clear_session()

    rows = [
        {'fold': CURRENT_FOLD, 'arch': arch_code, 'selected_by': 'val_auc', **reloaded_auc_metrics},
        {'fold': CURRENT_FOLD, 'arch': arch_code, 'selected_by': 'val_accuracy', **reloaded_acc_metrics},
    ]
    out_df = pd.DataFrame(rows)
    out_path = f'/kaggle/working/cv_f{CURRENT_FOLD}_{arch_code}_results.csv'
    out_df.to_csv(out_path, index=False)
    print(f"\nSaved {out_path}")
    print(out_df.round(4).to_string(index=False))
    return out_df

# ---- MOBILENETV2 ----
mob_results = train_and_evaluate('mob', MobileNetV2, mob_pre)

# ---- RESNET50 ----
res_results = train_and_evaluate('res', ResNet50, res_pre)

print(f"\n\n{'='*20} FOLD {CURRENT_FOLD}: MOB + RES DONE {'='*20}")
print(pd.concat([mob_results, res_results], ignore_index=True).round(4).to_string(index=False))
print(f"\nDownload individually: cv_f{CURRENT_FOLD}_mob_results.csv, cv_f{CURRENT_FOLD}_res_results.csv")
print(f"\nFOLD {CURRENT_FOLD} FULLY COMPLETE: custom, eff, mob, res all done, 4 of 4.")
print(f"Next: fold 2, all four architectures, same locked fold assignment file, CURRENT_FOLD=2.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 645.0/645.0 MB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 72.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
ydf-tf 2.20.0 requires tensorflow==2.20.0, but you have tensorflow 2.19.0 which is incompatible.
tf-keras 2.20.0 requires tensorflow<2.21,>=2.20, but you have tensorflow 2.19.0 which is incompatible.
tensorflow-text 2.20.1 requires tensorflow<2.21,>=2.20.0, but you have tensorflow 2.19.0 which is incompatible.


2026-08-25 02:58:55.748956: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1787626735.772037      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1787626735.779558      23 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1787626735.797734      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1787626735.797752      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1787626735.797754      23 computation_placer.cc:177] computation placer alr

Seed 42 set, TF 2.19.0, tf.keras module: tf_keras.api._v2.keras
Loaded 9,068 rows (expect 9,068)
fold
1    1811
2    1812
3    1816
4    1815
5    1814
Name: count, dtype: int64

Fold 1: Train 6,165 | Val 1,092 | Test 1,811
Must reproduce the custom/eff session exactly: Train 6,165 | Val 1,092 | Test 1,811
  aptos: train/val/test group-disjoint = True
  eyepacs: train/val/test group-disjoint = True
  messidor: train/val/test group-disjoint = True
Fold 1 leakage check: PASS
Fold 1 class_weight: {np.str_('0'): np.float64(0.33), np.str_('1'): np.float64(2.005), np.str_('2'): np.float64(0.951), np.str_('3'): np.float64(5.247), np.str_('4'): np.float64(4.404)} | span 15.9x
Must reproduce the custom/eff session: span 15.9x
Found 6165 validated image filenames belonging to 5 classes.
Found 1092 validated image filenames belonging to 5 classes.
Found 1811 validated image filenames belonging to 5 classes.


I0000 00:00:1787626763.777047      23 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1787626763.782998      23 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


9406464/9406464 [==============================] - 1s 0us/step

===== Fold 1, mob: PHASE 1 (head only, 10 epochs) =====
Epoch 1/10


I0000 00:00:1787626773.908240      75 cuda_dnn.cc:529] Loaded cuDNN version 91002
I0000 00:00:1787626776.541914      73 service.cc:152] XLA service 0x79199d2fb980 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1787626776.541947      73 service.cc:160]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1787626776.541950      73 service.cc:160]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1787626776.687756      73 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


193/193 [==============================] - 504s 3s/step - loss: 1.4818 - accuracy: 0.4337 - auc: 0.7515 - val_loss: 1.2263 - val_accuracy: 0.5366 - val_auc: 0.8179
Epoch 2/10
193/193 [==============================] - 361s 2s/step - loss: 1.2874 - accuracy: 0.5238 - auc: 0.8233 - val_loss: 1.1445 - val_accuracy: 0.5284 - val_auc: 0.8363
Epoch 3/10
193/193 [==============================] - 360s 2s/step - loss: 1.2440 - accuracy: 0.5530 - auc: 0.8365 - val_loss: 1.0396 - val_accuracy: 0.6145 - val_auc: 0.8743
Epoch 4/10
193/193 [==============================] - 362s 2s/step - loss: 1.2080 - accuracy: 0.5419 - auc: 0.8429 - val_loss: 1.0014 - val_accuracy: 0.6667 - val_auc: 0.8907
Epoch 5/10
193/193 [==============================] - 361s 2s/step - loss: 1.1661 - accuracy: 0.5792 - auc: 0.8547 - val_loss: 0.8703 - val_accuracy: 0.6813 - val_auc: 0.9142
Epoch 6/10
193/193 [==============================] - 357s 2s/step - loss: 1.1615 - accuracy: 0.5517 - auc: 0.8514 - val_loss: 1.0518 - 